# 05 — Feature Engineering

In this notebook, we prepare the cleaned employee attrition dataset for machine learning:
1. Separate target (`Attrition`) and independent features
2. Encode the target variable (`Yes` -> 1, `No` -> 0)
3. Identify numerical and categorical features
4. Perform stratified train-test split (80% train, 20% test)
5. Build a preprocessing pipeline (`StandardScaler` for numeric, `OneHotEncoder` for categoricals)
6. Save processed feature matrices and preprocessor artifact

In [1]:
import os
import pandas as pd
import numpy as np

DATA_PATH = '../data/processed/employee_attrition_cleaned.csv'
if not os.path.exists(DATA_PATH):
    DATA_PATH = 'data/processed/employee_attrition_cleaned.csv'

df = pd.read_csv(DATA_PATH)
print("Cleaned dataset shape:", df.shape)
df.head()

Cleaned dataset shape: (1470, 31)


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EnvironmentSatisfaction,Gender,...,PerformanceRating,RelationshipSatisfaction,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,2,Female,...,3,1,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,3,Male,...,4,4,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,4,Male,...,3,2,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,4,Female,...,3,3,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,Male,...,3,4,1,6,3,3,2,2,2,2


## 1. Encode Target Variable

In [2]:
# Target encoding: Yes -> 1 (Attrited), No -> 0 (Retained)
df['Attrition'] = (df['Attrition'] == 'Yes').astype(int)

print("Target distribution counts:")
print(df['Attrition'].value_counts())
print("\nTarget distribution percentage:")
print((df['Attrition'].value_counts(normalize=True) * 100).round(2))

Target distribution counts:
Attrition
0    1233
1     237
Name: count, dtype: int64

Target distribution percentage:
Attrition
0    83.88
1    16.12
Name: proportion, dtype: float64


## 2. Separate Features and Target

In [3]:
X = df.drop('Attrition', axis=1)
y = df['Attrition']

num_cols = X.select_dtypes(exclude='object').columns.tolist()
cat_cols = X.select_dtypes(include='object').columns.tolist()

print(f"Total features: {X.shape[1]}")
print(f"Numerical features ({len(num_cols)}):", num_cols[:5], "...")
print(f"Categorical features ({len(cat_cols)}):", cat_cols)

Total features: 30
Numerical features (23): ['Age', 'DailyRate', 'DistanceFromHome', 'Education', 'EnvironmentSatisfaction'] ...
Categorical features (7): ['BusinessTravel', 'Department', 'EducationField', 'Gender', 'JobRole', 'MaritalStatus', 'OverTime']


## 3. Stratified Train-Test Split (80/20)

In [4]:
from sklearn.model_selection import train_test_split

# Stratified split ensures the ~16% positive ratio is preserved across both splits
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training samples: {X_train.shape[0]}")
print(f"Testing samples:  {X_test.shape[0]}")
print("\nTrain target ratio:\n", y_train.value_counts(normalize=True).round(3))
print("\nTest target ratio:\n", y_test.value_counts(normalize=True).round(3))

Training samples: 1176
Testing samples:  294

Train target ratio:
 Attrition
0    0.838
1    0.162
Name: proportion, dtype: float64

Test target ratio:
 Attrition
0    0.84
1    0.16
Name: proportion, dtype: float64


## 4. Preprocessing: Scaling & One-Hot Encoding

In [5]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# StandardScaler normalizes numerical ranges; OneHotEncoder encodes nominal categories
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cat_cols)
    ]
)

# Fit exclusively on training data to prevent data leakage
X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)

# Retrieve encoded feature names
cat_encoder = preprocessor.named_transformers_['cat']
cat_encoded_names = cat_encoder.get_feature_names_out(cat_cols).tolist()
feature_names = num_cols + cat_encoded_names

print("Processed feature matrix shape:", X_train_proc.shape)
print("Total encoded features:", len(feature_names))

Processed feature matrix shape: (1176, 44)
Total encoded features: 44


## 5. Save Processed Datasets & Metadata

In [6]:
import joblib

data_dir = '../data/processed' if os.path.exists('../data/processed') else 'data/processed'
models_dir = '../models' if os.path.exists('../models') else 'models'

X_train_df = pd.DataFrame(X_train_proc, columns=feature_names)
X_test_df = pd.DataFrame(X_test_proc, columns=feature_names)

X_train_df.to_csv(f'{data_dir}/X_train.csv', index=False)
X_test_df.to_csv(f'{data_dir}/X_test.csv', index=False)
y_train.to_csv(f'{data_dir}/y_train.csv', index=False)
y_test.to_csv(f'{data_dir}/y_test.csv', index=False)

# Compute default median/mode values for raw inputs (useful for inference / Streamlit UI)
defaults = {}
for col in X.columns:
    if col in cat_cols:
        defaults[col] = X[col].mode()[0]
    else:
        defaults[col] = X[col].median()

joblib.dump({
    'preprocessor': preprocessor,
    'num_cols': num_cols,
    'cat_cols': cat_cols,
    'feature_names': feature_names,
    'defaults': defaults
}, f'{models_dir}/preprocessor.pkl')

print("Feature engineering completed successfully!")

Feature engineering completed successfully!
